## Cell 1 — Mount Drive and configure paths

In [1]:
# ============================================================
# Step 1.1: Environment and project paths
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from google.colab import drive


drive.mount("/content/drive")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/"
    "Amazon-Marketplace-Product-Intelligence-Platform-BJIT"
)

REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "price_tier"
)

INTERIM_DIR = (
    PROJECT_ROOT
    / "data"
    / "interim"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

INTERIM_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Project exists:", PROJECT_ROOT.exists())
print("Project path:", PROJECT_ROOT)

Mounted at /content/drive
Project exists: True
Project path: /content/drive/MyDrive/Amazon-Marketplace-Product-Intelligence-Platform-BJIT


## Cell 2 — Locate and load the product dataset

In [2]:
# ============================================================
# Step 1.2: Locate and load product data
# ============================================================

DATASET_CANDIDATES = [
    PROJECT_ROOT
    / "data"
    / "raw"
    / "amazon_products.csv",

    PROJECT_ROOT
    / "data"
    / "processed"
    / "products_recommendation_clean.csv",

    PROJECT_ROOT
    / "data"
    / "amazon_products.csv",
]

DATASET_PATH = next(
    (
        path
        for path in DATASET_CANDIDATES
        if path.exists()
    ),
    None,
)

if DATASET_PATH is None:
    raise FileNotFoundError(
        "Product dataset was not found. Checked:\n"
        + "\n".join(
            str(path)
            for path in DATASET_CANDIDATES
        )
    )

products_raw = pd.read_csv(
    DATASET_PATH,
    low_memory=False,
)

products_raw.columns = (
    products_raw.columns
    .astype(str)
    .str.strip()
)

print("Dataset:", DATASET_PATH)
print("Rows:", len(products_raw))
print("Columns:", len(products_raw.columns))
print("\nAvailable columns:")

for column in products_raw.columns:
    print(" -", column)

display(
    products_raw.head(3)
)

Dataset: /content/drive/MyDrive/Amazon-Marketplace-Product-Intelligence-Platform-BJIT/data/raw/amazon_products.csv
Rows: 1056
Columns: 12

Available columns:
 - asin
 - keyword
 - product_url
 - title
 - price
 - brand
 - description
 - image_url
 - video_url
 - avg_rating
 - rating_count
 - reviews


,asin,keyword,product_url,title,price,brand,description,image_url,video_url,avg_rating,rating_count,reviews
0,B0H9LHWBDK,wireless earbuds,https://www.amazon.com/dp/B0H9LHWBDK,"Wireless Earbuds, 2026 Bluetooth 5.4 Headphone...",$18.99,Brand: XIAOWTEK,LED Power Display and 50H Playback: Dual digit...,https://m.media-amazon.com/images/I/61odQgshCW...,https://m.media-amazon.com/images/S/vse-vms-tr...,4.8 out of 5 stars,(771),"[{""review_id"": ""R2CEXJGANFZ5NU"", ""reviewer_nam..."
1,B0HBWW3958,wireless earbuds,https://www.amazon.com/dp/B0HBWW3958,"Wireless Earbuds, Bluetooth 5.3 Headphones HiF...",$19.98,Brand: Fhumsh,Smart LED Display & 50H Sport Headphones: The ...,https://m.media-amazon.com/images/I/61D5ZCSIym...,https://m.media-amazon.com/images/S/vse-vms-tr...,4.3 out of 5 stars,(194),"[{""review_id"": ""R3N8ZPVF7TPV84"", ""reviewer_nam..."
2,B0H9RMD97X,wireless earbuds,https://www.amazon.com/dp/B0H9RMD97X,Air Pods for iPhone Bluetooth Wireless Ear Bud...,$13.99,Visit the Bioonis Store,【Balanced Hi-Fi Stereo Sound Earbuds】Experienc...,https://m.media-amazon.com/images/I/51wfpLrC24...,https://m.media-amazon.com/images/S/al-na-9d57...,4.3 out of 5 stars,NaN,[]


## Cell 3 — Identify the required columns

This handles slightly different column names automatically.

In [3]:
# ============================================================
# Step 1.3: Identify required columns
# ============================================================

COLUMN_CANDIDATES = {
    "asin": [
        "asin",
        "product_asin",
    ],
    "title": [
        "title",
        "product_title",
    ],
    "description": [
        "description",
        "product_description",
        "about_this_item",
    ],
    "brand": [
        "brand",
        "product_brand",
    ],
    "keyword": [
        "keyword_list",
        "search_keyword",
        "keyword",
    ],
    "domain": [
        "domain",
        "marketplace",
        "source_domain",
        "amazon_domain",
    ],
    "product_url": [
        "product_url",
        "url",
    ],
    "price": [
        "price",
        "current_price",
        "product_price",
    ],
    "review_count": [
        "review_count",
        "rating_count",
        "ratings_count",
        "number_of_reviews",
        "total_reviews",
    ],
    "average_rating": [
        "avg_rating",
        "average_rating",
        "rating",
        "star_rating",
    ],
}


def find_column(candidates):
    normalized_columns = {
        column.lower(): column
        for column in products_raw.columns
    }

    for candidate in candidates:
        if candidate.lower() in normalized_columns:
            return normalized_columns[
                candidate.lower()
            ]

    return None


column_map = {
    feature: find_column(candidates)
    for feature, candidates
    in COLUMN_CANDIDATES.items()
}

column_mapping_table = pd.DataFrame(
    {
        "required_feature": column_map.keys(),
        "detected_column": column_map.values(),
    }
)

display(column_mapping_table)

if column_map["price"] is None:
    raise ValueError(
        "A price column could not be identified."
    )

if column_map["asin"] is None:
    raise ValueError(
        "An ASIN column could not be identified."
    )

,required_feature,detected_column
0,asin,asin
1,title,title
2,description,description
3,brand,brand
4,keyword,keyword
5,domain,None
6,product_url,product_url
7,price,price
8,review_count,rating_count
9,average_rating,avg_rating


## Cell 4 — Parse price, review count and rating

In [4]:
# ============================================================
# Step 1.4: Audit numeric product attributes
# ============================================================

MISSING_TEXT_VALUES = {
    "",
    "nan",
    "none",
    "null",
    "na",
    "n/a",
    "unknown",
    "not available",
}


def extract_first_number(series):
    normalized = (
        series
        .astype("string")
        .str.replace(",", "", regex=False)
        .str.replace("\u00a0", " ", regex=False)
        .str.strip()
    )

    extracted = normalized.str.extract(
        r"([-+]?\d+(?:\.\d+)?)",
        expand=False,
    )

    return pd.to_numeric(
        extracted,
        errors="coerce",
    )


audit_data = pd.DataFrame(
    index=products_raw.index
)


# ------------------------------------------------------------
# ASIN
# ------------------------------------------------------------

audit_data["asin"] = (
    products_raw[column_map["asin"]]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)


# ------------------------------------------------------------
# Price
# ------------------------------------------------------------

raw_price = products_raw[
    column_map["price"]
]

parsed_price = extract_first_number(
    raw_price
)

audit_data["raw_price"] = raw_price
audit_data["parsed_price"] = parsed_price

audit_data["valid_price"] = (
    parsed_price.where(
        parsed_price > 0
    )
)

audit_data["invalid_or_missing_price"] = (
    audit_data["valid_price"].isna()
)

audit_data["negative_price"] = (
    parsed_price < 0
)


def detect_currency(value):
    if pd.isna(value):
        return "Unknown"

    value = str(value)

    if "£" in value:
        return "GBP"

    if "$" in value:
        return "USD"

    if "€" in value:
        return "EUR"

    return "Unspecified"


audit_data["detected_currency"] = (
    raw_price.map(detect_currency)
)


# ------------------------------------------------------------
# Review count
# ------------------------------------------------------------

if column_map["review_count"] is not None:
    raw_review_count = products_raw[
        column_map["review_count"]
    ]

    signed_review_count = (
        extract_first_number(
            raw_review_count
        )
    )

    audit_data["raw_review_count"] = (
        raw_review_count
    )

    audit_data[
        "review_count_had_negative_sign"
    ] = signed_review_count < 0

    # The negative sign is a scraping-format issue.
    audit_data["clean_review_count"] = (
        signed_review_count.abs()
    )
else:
    audit_data["raw_review_count"] = np.nan
    audit_data[
        "review_count_had_negative_sign"
    ] = False
    audit_data["clean_review_count"] = np.nan


# ------------------------------------------------------------
# Average rating
# ------------------------------------------------------------

if column_map["average_rating"] is not None:
    raw_average_rating = products_raw[
        column_map["average_rating"]
    ]

    parsed_average_rating = (
        extract_first_number(
            raw_average_rating
        )
    )

    audit_data["raw_average_rating"] = (
        raw_average_rating
    )

    audit_data["clean_average_rating"] = (
        parsed_average_rating.where(
            parsed_average_rating.between(
                0,
                5,
                inclusive="both",
            )
        )
    )
else:
    audit_data["raw_average_rating"] = np.nan
    audit_data["clean_average_rating"] = np.nan


numeric_audit = pd.DataFrame(
    {
        "metric": [
            "Total product rows",
            "Unique non-empty ASINs",
            "Duplicate ASIN rows",
            "Valid prices",
            "Missing or invalid prices",
            "Negative price values",
            "Review counts with negative sign",
            "Missing review counts after parsing",
            "Valid average ratings",
            "Missing or invalid average ratings",
        ],
        "value": [
            len(audit_data),
            audit_data.loc[
                audit_data["asin"].ne(""),
                "asin",
            ].nunique(),
            int(
                audit_data.loc[
                    audit_data["asin"].ne(""),
                    "asin",
                ].duplicated().sum()
            ),
            int(
                audit_data["valid_price"]
                .notna()
                .sum()
            ),
            int(
                audit_data[
                    "invalid_or_missing_price"
                ].sum()
            ),
            int(
                audit_data[
                    "negative_price"
                ].sum()
            ),
            int(
                audit_data[
                    "review_count_had_negative_sign"
                ].sum()
            ),
            int(
                audit_data[
                    "clean_review_count"
                ].isna()
                .sum()
            ),
            int(
                audit_data[
                    "clean_average_rating"
                ].notna()
                .sum()
            ),
            int(
                audit_data[
                    "clean_average_rating"
                ].isna()
                .sum()
            ),
        ],
    }
)

display(numeric_audit)

display(
    audit_data[
        [
            "raw_price",
            "valid_price",
            "raw_review_count",
            "clean_review_count",
            "raw_average_rating",
            "clean_average_rating",
        ]
    ].head(10)
)

,metric,value
0,Total product rows,1056
1,Unique non-empty ASINs,981
2,Duplicate ASIN rows,75
3,Valid prices,984
4,Missing or invalid prices,72
5,Negative price values,0
6,Review counts with negative sign,0
7,Missing review counts after parsing,9
8,Valid average ratings,1048
9,Missing or invalid average ratings,8


,raw_price,valid_price,raw_review_count,clean_review_count,raw_average_rating,clean_average_rating
0,$18.99,18.99,(771),771,4.8 out of 5 stars,4.8
1,$19.98,19.98,(194),194,4.3 out of 5 stars,4.3
2,$13.99,13.99,NaN,<NA>,4.3 out of 5 stars,4.3
3,$16.99,16.99,(353),353,4.3 out of 5 stars,4.3
4,$23.98,23.98,(351),351,4.8 out of 5 stars,4.8
5,$23.98,23.98,(352),352,4.8 out of 5 stars,4.8
6,$35.99,35.99,"(1,709)",1709,4.0 out of 5 stars,4.0
7,$19.99,19.99,(99),99,4.8 out of 5 stars,4.8
8,$16.99,16.99,(74),74,5 out of 5 stars,5.0
9,$20.99,20.99,(129),129,4.8 out of 5 stars,4.8


## Cell 5 — Audit missing and problematic text

This cell only identifies text problems. It does not remove valid Unicode or emojis

In [5]:
# ============================================================
# Step 1.5: Audit missing and problematic text
# ============================================================

TEXT_FEATURES = [
    "title",
    "description",
    "brand",
    "keyword",
]


def audit_text_column(
    feature_name,
    column_name,
):
    if column_name is None:
        return {
            "feature": feature_name,
            "detected_column": None,
            "missing_count": len(products_raw),
            "repeated_whitespace_count": 0,
            "replacement_character_count": 0,
            "control_character_count": 0,
            "non_ascii_unicode_count": 0,
        }

    text = (
        products_raw[column_name]
        .astype("string")
    )

    stripped = (
        text
        .fillna("")
        .str.strip()
    )

    missing_mask = (
        text.isna()
        | stripped.str.lower().isin(
            MISSING_TEXT_VALUES
        )
    )

    repeated_whitespace = (
        stripped.str.contains(
            r"\s{2,}|\t|\r|\n",
            regex=True,
            na=False,
        )
    )

    replacement_character = (
        stripped.str.contains(
            "\ufffd",
            regex=False,
            na=False,
        )
    )

    control_character = (
        stripped.str.contains(
            r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]",
            regex=True,
            na=False,
        )
    )

    non_ascii_unicode = stripped.map(
        lambda value: any(
            ord(character) > 127
            for character in value
        )
    )

    return {
        "feature": feature_name,
        "detected_column": column_name,
        "missing_count": int(
            missing_mask.sum()
        ),
        "repeated_whitespace_count": int(
            repeated_whitespace.sum()
        ),
        "replacement_character_count": int(
            replacement_character.sum()
        ),
        "control_character_count": int(
            control_character.sum()
        ),
        "non_ascii_unicode_count": int(
            non_ascii_unicode.sum()
        ),
    }


text_quality_audit = pd.DataFrame(
    [
        audit_text_column(
            feature,
            column_map[feature],
        )
        for feature in TEXT_FEATURES
    ]
)

display(text_quality_audit)

,feature,detected_column,missing_count,repeated_whitespace_count,replacement_character_count,control_character_count,non_ascii_unicode_count
0,title,title,1,0,0,0,59
1,description,description,4,0,0,0,651
2,brand,brand,30,0,0,0,6
3,keyword,keyword,0,0,0,0,0


## Cell 6 — Audit comparable price groups

In [6]:
# ============================================================
# Step 1.6: Audit marketplace-keyword groups
# ============================================================

grouping_data = pd.DataFrame(
    {
        "asin": audit_data["asin"],
        "price": audit_data["valid_price"],
    }
)


# Marketplace/domain
if column_map["domain"] is not None:
    grouping_data["marketplace"] = (
        products_raw[
            column_map["domain"]
        ]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
        .replace("", "Unknown")
    )

elif column_map["product_url"] is not None:
    product_urls = (
        products_raw[
            column_map["product_url"]
        ]
        .fillna("")
        .astype(str)
        .str.lower()
    )

    grouping_data["marketplace"] = np.select(
        [
            product_urls.str.contains(
                "amazon.co.uk",
                regex=False,
            ),
            product_urls.str.contains(
                "amazon.com",
                regex=False,
            ),
        ],
        [
            "amazon_uk",
            "amazon_us",
        ],
        default="Unknown",
    )

else:
    grouping_data["marketplace"] = (
        "all_marketplaces"
    )


# Search keyword
if column_map["keyword"] is not None:
    grouping_data["keyword"] = (
        products_raw[
            column_map["keyword"]
        ]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
        .replace("", "Unknown")
    )
else:
    grouping_data["keyword"] = "Unknown"


MINIMUM_VALID_PRICES = 15

price_group_audit = (
    grouping_data
    .groupby(
        [
            "marketplace",
            "keyword",
        ],
        dropna=False,
    )
    .agg(
        product_count=(
            "asin",
            "size",
        ),
        valid_price_count=(
            "price",
            "count",
        ),
        unique_price_count=(
            "price",
            "nunique",
        ),
        minimum_price=(
            "price",
            "min",
        ),
        median_price=(
            "price",
            "median",
        ),
        maximum_price=(
            "price",
            "max",
        ),
    )
    .reset_index()
)

price_group_audit["missing_price_count"] = (
    price_group_audit["product_count"]
    - price_group_audit["valid_price_count"]
)

price_group_audit["labeling_status"] = np.where(
    (
        price_group_audit[
            "valid_price_count"
        ].ge(MINIMUM_VALID_PRICES)
        & price_group_audit[
            "unique_price_count"
        ].ge(3)
    ),
    "Ready for group thresholds",
    "Needs fallback threshold",
)

price_group_audit = (
    price_group_audit
    .sort_values(
        [
            "labeling_status",
            "valid_price_count",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)

display(price_group_audit)

display(
    price_group_audit[
        "labeling_status"
    ]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="group_count")
)

,marketplace,keyword,product_count,valid_price_count,unique_price_count,minimum_price,median_price,maximum_price,missing_price_count,labeling_status
0,amazon_us,sunglasses,62,62,39,6.98,15.985,289.0,0,Ready for group thresholds
1,amazon_us,desk lamp LED,60,60,34,9.99,21.485,69.99,0,Ready for group thresholds
2,amazon_us,yamaha accoustic guitar,60,59,43,6.49,71.99,33867.0,1,Ready for group thresholds
3,amazon_us,kitchen knife set,60,58,44,15.98,58.985,399.0,2,Ready for group thresholds
4,amazon_us,sneakers,60,58,48,18.49,50.98,162.8,2,Ready for group thresholds
5,amazon_us,notebook journal,59,56,35,3.99,9.99,84.99,3,Ready for group thresholds
6,amazon_us,water bottle,62,55,39,8.99,19.99,522.0,7,Ready for group thresholds
7,amazon_us,electric toothbrush,60,54,34,8.98,39.99,299.0,6,Ready for group thresholds
8,amazon_us,gaming chair,60,52,31,25.99,119.98,299.0,8,Ready for group thresholds
9,amazon_us,candle,56,51,37,9.99,22.89,90.0,5,Ready for group thresholds


,status,group_count
0,Ready for group thresholds,28


In [7]:
# ============================================================
# Step 1.7: Save audit reports
# ============================================================

NUMERIC_AUDIT_PATH = (
    REPORT_DIR
    / "price_data_audit.csv"
)

TEXT_AUDIT_PATH = (
    REPORT_DIR
    / "price_text_quality_audit.csv"
)

GROUP_AUDIT_PATH = (
    REPORT_DIR
    / "price_group_audit.csv"
)

AUDIT_DATA_PATH = (
    INTERIM_DIR
    / "price_tier_audit_data.csv"
)


numeric_audit.to_csv(
    NUMERIC_AUDIT_PATH,
    index=False,
    encoding="utf-8-sig",
)

text_quality_audit.to_csv(
    TEXT_AUDIT_PATH,
    index=False,
    encoding="utf-8-sig",
)

price_group_audit.to_csv(
    GROUP_AUDIT_PATH,
    index=False,
    encoding="utf-8-sig",
)

audit_data.to_csv(
    AUDIT_DATA_PATH,
    index=False,
    encoding="utf-8-sig",
)


print("Step 1 completed successfully.")
print("Numeric audit:", NUMERIC_AUDIT_PATH)
print("Text audit:", TEXT_AUDIT_PATH)
print("Group audit:", GROUP_AUDIT_PATH)
print("Parsed audit data:", AUDIT_DATA_PATH)

Step 1 completed successfully.
Numeric audit: /content/drive/MyDrive/Amazon-Marketplace-Product-Intelligence-Platform-BJIT/reports/price_tier/price_data_audit.csv
Text audit: /content/drive/MyDrive/Amazon-Marketplace-Product-Intelligence-Platform-BJIT/reports/price_tier/price_text_quality_audit.csv
Group audit: /content/drive/MyDrive/Amazon-Marketplace-Product-Intelligence-Platform-BJIT/reports/price_tier/price_group_audit.csv
Parsed audit data: /content/drive/MyDrive/Amazon-Marketplace-Product-Intelligence-Platform-BJIT/data/interim/price_tier_audit_data.csv


In [8]:
# ============================================================
# Step 2.1: Clean product attributes
# ============================================================

import html
import re
import unicodedata


if "products_raw" not in globals():
    raise NameError(
        "products_raw is unavailable. "
        "Rerun Step 1 Cells 1–3."
    )

if "column_map" not in globals():
    raise NameError(
        "column_map is unavailable. "
        "Rerun Step 1 Cell 3."
    )


MISSING_VALUES = {
    "",
    "nan",
    "none",
    "null",
    "na",
    "n/a",
    "unknown",
    "not available",
}

HTML_TAG_PATTERN = re.compile(
    r"<[^>]+>"
)

CONTROL_CHARACTER_PATTERN = re.compile(
    r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]"
)

REPEATED_WHITESPACE_PATTERN = re.compile(
    r"\s+"
)


def get_raw_series(
    feature_name,
    default="",
):
    column_name = column_map.get(
        feature_name
    )

    if column_name is None:
        return pd.Series(
            default,
            index=products_raw.index,
        )

    return products_raw[
        column_name
    ]


def clean_text(value):
    """
    Clean corrupted and unnecessary formatting while
    preserving valid Unicode, punctuation and emojis.
    """

    if pd.isna(value):
        return ""

    text = str(value)

    text = html.unescape(text)

    text = unicodedata.normalize(
        "NFKC",
        text,
    )

    text = text.replace(
        "\ufffd",
        " ",
    )

    text = HTML_TAG_PATTERN.sub(
        " ",
        text,
    )

    text = CONTROL_CHARACTER_PATTERN.sub(
        " ",
        text,
    )

    text = REPEATED_WHITESPACE_PATTERN.sub(
        " ",
        text,
    ).strip()

    if text.lower() in MISSING_VALUES:
        return ""

    return text


def extract_first_number(series):
    normalized = (
        series
        .astype("string")
        .str.replace(",", "", regex=False)
        .str.replace("\u00a0", " ", regex=False)
        .str.strip()
    )

    extracted = normalized.str.extract(
        r"([-+]?\d+(?:\.\d+)?)",
        expand=False,
    )

    return pd.to_numeric(
        extracted,
        errors="coerce",
    )


clean_products = pd.DataFrame(
    index=products_raw.index
)


# ASIN
clean_products["asin"] = (
    get_raw_series("asin")
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)


# Text features
clean_products["title"] = (
    get_raw_series("title")
    .map(clean_text)
)

clean_products["description"] = (
    get_raw_series("description")
    .map(clean_text)
)

clean_products["brand"] = (
    get_raw_series("brand")
    .map(clean_text)
)

clean_products["keyword"] = (
    get_raw_series("keyword")
    .map(clean_text)
)


# Price
raw_price = get_raw_series(
    "price",
    default=np.nan,
)

parsed_price = extract_first_number(
    raw_price
)

clean_products["price"] = (
    parsed_price.where(
        parsed_price > 0
    )
)


# Review count
raw_review_count = get_raw_series(
    "review_count",
    default=np.nan,
)

parsed_review_count = (
    extract_first_number(
        raw_review_count
    )
)

# Negative signs are a scraping-format issue.
clean_products["review_count"] = (
    parsed_review_count.abs()
)

clean_products["review_count_missing"] = (
    clean_products["review_count"]
    .isna()
    .astype(int)
)


# Average rating
raw_average_rating = get_raw_series(
    "average_rating",
    default=np.nan,
)

parsed_average_rating = (
    extract_first_number(
        raw_average_rating
    )
)

clean_products["average_rating"] = (
    parsed_average_rating.where(
        parsed_average_rating.between(
            0,
            5,
            inclusive="both",
        )
    )
)

clean_products["rating_missing"] = (
    clean_products["average_rating"]
    .isna()
    .astype(int)
)


# Missing text indicators
clean_products["title_missing"] = (
    clean_products["title"]
    .eq("")
    .astype(int)
)

clean_products["description_missing"] = (
    clean_products["description"]
    .eq("")
    .astype(int)
)

clean_products["brand_missing"] = (
    clean_products["brand"]
    .eq("")
    .astype(int)
)


# Replace missing categorical values.
clean_products["brand"] = (
    clean_products["brand"]
    .replace("", "Unknown")
)

clean_products["keyword"] = (
    clean_products["keyword"]
    .replace("", "Unknown")
)


print("Initial rows:", len(clean_products))
print(
    "Review counts converted to non-negative:",
    int(
        clean_products["review_count"]
        .dropna()
        .ge(0)
        .all()
    ),
)

display(
    clean_products.head()
)

Initial rows: 1056
Review counts converted to non-negative: 1


,asin,title,description,brand,keyword,price,review_count,review_count_missing,average_rating,rating_missing,title_missing,description_missing,brand_missing
0,B0H9LHWBDK,"Wireless Earbuds, 2026 Bluetooth 5.4 Headphone...",LED Power Display and 50H Playback: Dual digit...,Brand: XIAOWTEK,wireless earbuds,18.99,771,0,4.8,0,0,0,0
1,B0HBWW3958,"Wireless Earbuds, Bluetooth 5.3 Headphones HiF...",Smart LED Display & 50H Sport Headphones: The ...,Brand: Fhumsh,wireless earbuds,19.98,194,0,4.3,0,0,0,0
2,B0H9RMD97X,Air Pods for iPhone Bluetooth Wireless Ear Bud...,【Balanced Hi-Fi Stereo Sound Earbuds】Experienc...,Visit the Bioonis Store,wireless earbuds,13.99,<NA>,1,4.3,0,0,0,0
3,B0G7H6K1WN,"Wireless Earbuds,Bluetooth 5.4 Ear buds,Sport ...","Secure & Comfortable Earhook Fit – Flexible, e...",Visit the KTGEE Store,wireless earbuds,16.99,353,0,4.3,0,0,0,0
4,B0H7R1LD7B,"HAOYUYAN Wireless Earbuds, Sports Bluetooth He...",【Sports Comfort & IPX7 Waterproof】BX17 Wireles...,Visit the HAOYUYAN Store,wireless earbuds,23.98,351,0,4.8,0,0,0,0


## Cell 9 — Create the marketplace field

In [9]:
# ============================================================
# Step 2.2: Create marketplace/domain field
# ============================================================

if column_map.get("domain") is not None:
    clean_products["marketplace"] = (
        get_raw_series("domain")
        .map(clean_text)
        .replace("", "Unknown")
    )

else:
    product_urls = (
        get_raw_series("product_url")
        .fillna("")
        .astype(str)
        .str.lower()
    )

    clean_products["marketplace"] = np.select(
        [
            product_urls.str.contains(
                "amazon.co.uk",
                regex=False,
            ),
            product_urls.str.contains(
                "amazon.com",
                regex=False,
            ),
        ],
        [
            "amazon_uk",
            "amazon_us",
        ],
        default="Unknown",
    )


def detect_currency(value):
    if pd.isna(value):
        return "Unknown"

    value = str(value)

    if "£" in value:
        return "GBP"

    if "$" in value:
        return "USD"

    if "€" in value:
        return "EUR"

    return "Unspecified"


clean_products["currency"] = (
    raw_price.map(detect_currency)
)


# Use currency when marketplace is unavailable.
unknown_marketplace = (
    clean_products["marketplace"]
    .astype(str)
    .str.lower()
    .isin(
        {
            "",
            "unknown",
            "nan",
            "none",
        }
    )
)

clean_products.loc[
    unknown_marketplace
    & clean_products["currency"].eq("USD"),
    "marketplace",
] = "amazon_us"

clean_products.loc[
    unknown_marketplace
    & clean_products["currency"].eq("GBP"),
    "marketplace",
] = "amazon_uk"


# Remove rows without a usable ASIN.
empty_asin_count = int(
    clean_products["asin"]
    .eq("")
    .sum()
)

clean_products = clean_products[
    clean_products["asin"].ne("")
].copy()


# Prefer the most complete row when duplicate ASINs exist.
completeness_columns = [
    "title",
    "description",
    "brand",
    "keyword",
    "price",
    "review_count",
    "average_rating",
]

clean_products["_completeness_score"] = (
    clean_products[
        completeness_columns
    ]
    .notna()
    .sum(axis=1)
)

clean_products = (
    clean_products
    .sort_values(
        "_completeness_score",
        ascending=False,
    )
    .drop_duplicates(
        "asin",
        keep="first",
    )
    .drop(
        columns="_completeness_score"
    )
    .reset_index(drop=True)
)


print("Removed empty ASIN rows:", empty_asin_count)
print("Clean unique products:", len(clean_products))
print(
    "Unique ASINs:",
    clean_products["asin"].nunique(),
)

display(
    clean_products[
        [
            "asin",
            "title",
            "marketplace",
            "currency",
            "keyword",
            "price",
            "review_count",
            "average_rating",
        ]
    ].head()
)

Removed empty ASIN rows: 0
Clean unique products: 981
Unique ASINs: 981


,asin,title,marketplace,currency,keyword,price,review_count,average_rating
0,B003GMP03Q,Yamaha FS50BT Acoustic Guitar Strings,amazon_us,USD,yamaha accoustic guitar,15.77,360,4.7
1,B0CW6KY2ZK,"Pyle Beginner Acoustic Guitar Kit, 1/4 Junior ...",amazon_us,USD,yamaha accoustic guitar,72.99,51,4.3
2,B001U0OWTO,"HOHNER 6 String Acoustic Guitar, Right Handed,...",amazon_us,USD,yamaha accoustic guitar,44.99,1055,4.3
3,B07XJLD2BL,Yamaha THR30II Wireless Desktop Guitar Amp,amazon_us,USD,yamaha accoustic guitar,599.99,380,4.2
4,B09XVLKGFT,"Yamaha THR30II Wireless Desktop Guitar Amp, Wh...",amazon_us,USD,yamaha accoustic guitar,599.99,380,4.2


In [11]:
# ============================================================
# Step 2.3: Calculate tier thresholds
# ============================================================

MINIMUM_GROUP_PRICES = 15
MINIMUM_UNIQUE_PRICES = 3

labelable_products = clean_products[
    clean_products["price"].notna()
].copy()

if len(labelable_products) < 30:
    raise ValueError(
        "There are not enough valid prices to "
        "create reliable price tiers."
    )


def calculate_thresholds(
    dataframe,
    group_columns,
):
    return (
        dataframe
        .groupby(
            group_columns,
            dropna=False,
        )
        .agg(
            valid_price_count=(
                "price",
                "count",
            ),
            unique_price_count=(
                "price",
                "nunique",
            ),
            minimum_price=(
                "price",
                "min",
            ),
            budget_upper=(
                "price",
                lambda values: values.quantile(
                    1 / 3
                ),
            ),
            midrange_upper=(
                "price",
                lambda values: values.quantile(
                    2 / 3
                ),
            ),
            maximum_price=(
                "price",
                "max",
            ),
        )
        .reset_index()
    )


# Preferred thresholds: marketplace + keyword
group_thresholds = calculate_thresholds(
    labelable_products,
    [
        "marketplace",
        "keyword",
    ],
)

group_thresholds["group_ready"] = (
    group_thresholds[
        "valid_price_count"
    ].ge(MINIMUM_GROUP_PRICES)
    & group_thresholds[
        "unique_price_count"
    ].ge(MINIMUM_UNIQUE_PRICES)
    & group_thresholds[
        "budget_upper"
    ].lt(
        group_thresholds[
            "midrange_upper"
        ]
    )
)


# Fallback thresholds: marketplace
marketplace_thresholds = calculate_thresholds(
    labelable_products,
    ["marketplace"],
)

marketplace_thresholds[
    "marketplace_ready"
] = (
    marketplace_thresholds[
        "valid_price_count"
    ].ge(MINIMUM_GROUP_PRICES)
    & marketplace_thresholds[
        "unique_price_count"
    ].ge(MINIMUM_UNIQUE_PRICES)
    & marketplace_thresholds[
        "budget_upper"
    ].lt(
        marketplace_thresholds[
            "midrange_upper"
        ]
    )
)


# Final fallback: complete valid-price dataset
global_budget_upper = float(
    labelable_products["price"]
    .quantile(1 / 3)
)

global_midrange_upper = float(
    labelable_products["price"]
    .quantile(2 / 3)
)

if global_budget_upper >= global_midrange_upper:
    raise ValueError(
        "Global price thresholds are not distinct."
    )


display(
    group_thresholds.head(10)
)

display(
    marketplace_thresholds
)

print(
    "Global budget upper threshold:",
    round(global_budget_upper, 2),
)

print(
    "Global mid-range upper threshold:",
    round(global_midrange_upper, 2),
)

,marketplace,keyword,valid_price_count,unique_price_count,minimum_price,budget_upper,midrange_upper,maximum_price,group_ready
0,amazon_us,DSLR camera,15,14,17.5,93.783333,535.666667,79900.0,True
1,amazon_us,USB-C hub,17,16,8.94,16.646667,33.983333,79.99,True
2,amazon_us,air purifier,16,15,16.99,49.87,69.99,199.0,True
3,amazon_us,backpack,23,20,15.0,27.17,37.65,279.0,True
4,amazon_us,bluetooth speaker,15,13,14.97,21.323333,59.636667,179.99,True
5,amazon_us,budget laptop,20,19,149.99,258.786667,377.663333,1359.0,True
6,amazon_us,candle,48,37,9.99,19.31,25.99,90.0,True
7,amazon_us,coffee maker,46,36,13.46,59.99,109.95,1995.0,True
8,amazon_us,desk lamp LED,49,34,9.99,19.98,29.99,69.99,True
9,amazon_us,electric toothbrush,49,34,8.98,25.98,53.99,299.0,True


,marketplace,valid_price_count,unique_price_count,minimum_price,budget_upper,midrange_upper,maximum_price,marketplace_ready
0,amazon_us,913,429,3.98,20.99,59.98,79900.0,True


Global budget upper threshold: 20.99
Global mid-range upper threshold: 59.98


## Cell 11 — Apply the appropriate threshold to each product

In [12]:
# ============================================================
# Step 2.4: Generate Budget/Mid-range/Premium labels
# ============================================================

group_threshold_columns = {
    "valid_price_count": (
        "group_valid_price_count"
    ),
    "unique_price_count": (
        "group_unique_price_count"
    ),
    "budget_upper": (
        "group_budget_upper"
    ),
    "midrange_upper": (
        "group_midrange_upper"
    ),
}

group_threshold_lookup = (
    group_thresholds
    .rename(
        columns=group_threshold_columns
    )
)

clean_products = clean_products.merge(
    group_threshold_lookup[
        [
            "marketplace",
            "keyword",
            "group_valid_price_count",
            "group_unique_price_count",
            "group_budget_upper",
            "group_midrange_upper",
            "group_ready",
        ]
    ],
    on=[
        "marketplace",
        "keyword",
    ],
    how="left",
    validate="many_to_one",
)


marketplace_threshold_lookup = (
    marketplace_thresholds
    .rename(
        columns={
            "valid_price_count": (
                "market_valid_price_count"
            ),
            "unique_price_count": (
                "market_unique_price_count"
            ),
            "budget_upper": (
                "market_budget_upper"
            ),
            "midrange_upper": (
                "market_midrange_upper"
            ),
        }
    )
)

clean_products = clean_products.merge(
    marketplace_threshold_lookup[
        [
            "marketplace",
            "market_valid_price_count",
            "market_unique_price_count",
            "market_budget_upper",
            "market_midrange_upper",
            "marketplace_ready",
        ]
    ],
    on="marketplace",
    how="left",
    validate="many_to_one",
)


use_group_threshold = (
    clean_products["group_ready"]
    .fillna(False)
)

use_marketplace_threshold = (
    ~use_group_threshold
    & clean_products["marketplace_ready"]
    .fillna(False)
)


clean_products["budget_upper"] = np.select(
    [
        use_group_threshold,
        use_marketplace_threshold,
    ],
    [
        clean_products[
            "group_budget_upper"
        ],
        clean_products[
            "market_budget_upper"
        ],
    ],
    default=global_budget_upper,
)

clean_products["midrange_upper"] = np.select(
    [
        use_group_threshold,
        use_marketplace_threshold,
    ],
    [
        clean_products[
            "group_midrange_upper"
        ],
        clean_products[
            "market_midrange_upper"
        ],
    ],
    default=global_midrange_upper,
)


clean_products["threshold_source"] = np.select(
    [
        use_group_threshold,
        use_marketplace_threshold,
    ],
    [
        "marketplace_keyword",
        "marketplace",
    ],
    default="global",
)


clean_products["price_tier"] = pd.NA

valid_price_mask = (
    clean_products["price"].notna()
)

clean_products.loc[
    valid_price_mask,
    "price_tier",
] = np.select(
    [
        clean_products.loc[
            valid_price_mask,
            "price",
        ].le(
            clean_products.loc[
                valid_price_mask,
                "budget_upper",
            ]
        ),
        clean_products.loc[
            valid_price_mask,
            "price",
        ].le(
            clean_products.loc[
                valid_price_mask,
                "midrange_upper",
            ]
        ),
    ],
    [
        "Budget",
        "Mid-range",
    ],
    default="Premium",
)


TIER_ORDER = [
    "Budget",
    "Mid-range",
    "Premium",
]

clean_products["price_tier"] = pd.Categorical(
    clean_products["price_tier"],
    categories=TIER_ORDER,
    ordered=True,
)


display(
    clean_products[
        [
            "asin",
            "marketplace",
            "keyword",
            "price",
            "budget_upper",
            "midrange_upper",
            "threshold_source",
            "price_tier",
        ]
    ].head(15)
)

,asin,marketplace,keyword,price,budget_upper,midrange_upper,threshold_source,price_tier
0,B003GMP03Q,amazon_us,yamaha accoustic guitar,15.77,44.99,89.99,marketplace_keyword,Budget
1,B0CW6KY2ZK,amazon_us,yamaha accoustic guitar,72.99,44.99,89.99,marketplace_keyword,Mid-range
2,B001U0OWTO,amazon_us,yamaha accoustic guitar,44.99,44.99,89.99,marketplace_keyword,Mid-range
3,B07XJLD2BL,amazon_us,yamaha accoustic guitar,599.99,44.99,89.99,marketplace_keyword,Premium
4,B09XVLKGFT,amazon_us,yamaha accoustic guitar,599.99,44.99,89.99,marketplace_keyword,Premium
5,B0D2PNMDJZ,amazon_us,yamaha accoustic guitar,18.95,44.99,89.99,marketplace_keyword,Budget
6,B0H28ZM4NL,amazon_us,yamaha accoustic guitar,71.99,44.99,89.99,marketplace_keyword,Mid-range
7,B07FYQP9ML,amazon_us,yamaha accoustic guitar,13.6,44.99,89.99,marketplace_keyword,Budget
8,B0H4HQ9XFN,amazon_us,yamaha accoustic guitar,71.95,44.99,89.99,marketplace_keyword,Mid-range
9,B0849ZFBGS,amazon_us,yamaha accoustic guitar,62.99,44.99,89.99,marketplace_keyword,Mid-range


In [13]:
# ============================================================
# Step 2.5: Validate generated labels
# ============================================================

labeled_products = clean_products[
    clean_products["price_tier"].notna()
].copy()


tier_distribution = (
    labeled_products[
        "price_tier"
    ]
    .value_counts(
        sort=False
    )
    .reindex(
        TIER_ORDER,
        fill_value=0,
    )
    .rename("product_count")
    .reset_index()
    .rename(
        columns={
            "price_tier": "tier"
        }
    )
)

tier_distribution["percentage"] = (
    tier_distribution["product_count"]
    / len(labeled_products)
    * 100
)


group_tier_audit = (
    pd.crosstab(
        index=[
            labeled_products[
                "marketplace"
            ],
            labeled_products[
                "keyword"
            ],
        ],
        columns=labeled_products[
            "price_tier"
        ],
    )
    .reindex(
        columns=TIER_ORDER,
        fill_value=0,
    )
    .reset_index()
)

group_tier_audit[
    "all_tiers_present"
] = (
    group_tier_audit[
        TIER_ORDER
    ]
    .gt(0)
    .all(axis=1)
)


threshold_source_summary = (
    labeled_products[
        "threshold_source"
    ]
    .value_counts()
    .rename_axis(
        "threshold_source"
    )
    .reset_index(
        name="product_count"
    )
)


missing_feature_summary = pd.DataFrame(
    {
        "feature": [
            "Price",
            "Title",
            "Description",
            "Brand",
            "Review count",
            "Average rating",
        ],
        "missing_count": [
            int(
                clean_products[
                    "price"
                ].isna().sum()
            ),
            int(
                clean_products[
                    "title_missing"
                ].sum()
            ),
            int(
                clean_products[
                    "description_missing"
                ].sum()
            ),
            int(
                clean_products[
                    "brand_missing"
                ].sum()
            ),
            int(
                clean_products[
                    "review_count_missing"
                ].sum()
            ),
            int(
                clean_products[
                    "rating_missing"
                ].sum()
            ),
        ],
    }
)


print("Complete catalogue:", len(clean_products))
print("Products with tier labels:", len(labeled_products))
print(
    "Products without price labels:",
    int(
        clean_products["price_tier"]
        .isna()
        .sum()
    ),
)

print("\nTier distribution:")
display(tier_distribution)

print("\nThreshold sources:")
display(threshold_source_summary)

print("\nMissing input features:")
display(missing_feature_summary)

print(
    "\nGroups missing at least one tier:",
    int(
        (~group_tier_audit[
            "all_tiers_present"
        ]).sum()
    ),
)

Complete catalogue: 981
Products with tier labels: 913
Products without price labels: 68

Tier distribution:


,tier,product_count,percentage
0,Budget,312,34.173056
1,Mid-range,298,32.639650
2,Premium,303,33.187295



Threshold sources:


,threshold_source,product_count
0,marketplace_keyword,913



Missing input features:


,feature,missing_count
0,Price,68
1,Title,0
2,Description,3
3,Brand,29
4,Review count,8
5,Average rating,7



Groups missing at least one tier: 0


In [14]:
# ============================================================
# Step 2.6: Save cleaned data and tier-label reports
# ============================================================

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


CLEAN_DATA_PATH = (
    PROCESSED_DIR
    / "products_price_tier_clean.csv"
)

LABELED_DATA_PATH = (
    PROCESSED_DIR
    / "products_price_tier_labeled.csv"
)

THRESHOLD_PATH = (
    REPORT_DIR
    / "price_tier_thresholds.csv"
)

DISTRIBUTION_PATH = (
    REPORT_DIR
    / "price_tier_distribution.csv"
)

GROUP_TIER_AUDIT_PATH = (
    REPORT_DIR
    / "price_tier_group_label_audit.csv"
)

MISSING_FEATURE_PATH = (
    REPORT_DIR
    / "price_tier_missing_features.csv"
)


threshold_report = (
    clean_products[
        [
            "marketplace",
            "keyword",
            "threshold_source",
            "budget_upper",
            "midrange_upper",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "marketplace",
            "keyword",
        ]
    )
    .reset_index(drop=True)
)


columns_to_remove = [
    "group_valid_price_count",
    "group_unique_price_count",
    "group_budget_upper",
    "group_midrange_upper",
    "group_ready",
    "market_valid_price_count",
    "market_unique_price_count",
    "market_budget_upper",
    "market_midrange_upper",
    "marketplace_ready",
]

clean_output = clean_products.drop(
    columns=[
        column
        for column in columns_to_remove
        if column in clean_products.columns
    ]
)


clean_output.to_csv(
    CLEAN_DATA_PATH,
    index=False,
    encoding="utf-8-sig",
)

clean_output[
    clean_output["price_tier"].notna()
].to_csv(
    LABELED_DATA_PATH,
    index=False,
    encoding="utf-8-sig",
)

threshold_report.to_csv(
    THRESHOLD_PATH,
    index=False,
    encoding="utf-8-sig",
)

tier_distribution.to_csv(
    DISTRIBUTION_PATH,
    index=False,
    encoding="utf-8-sig",
)

group_tier_audit.to_csv(
    GROUP_TIER_AUDIT_PATH,
    index=False,
    encoding="utf-8-sig",
)

missing_feature_summary.to_csv(
    MISSING_FEATURE_PATH,
    index=False,
    encoding="utf-8-sig",
)


print("Step 2 completed successfully.")
print("Complete clean data:", CLEAN_DATA_PATH)
print("Labelled training data:", LABELED_DATA_PATH)
print("Threshold report:", THRESHOLD_PATH)
print("Tier distribution:", DISTRIBUTION_PATH)
print("Group label audit:", GROUP_TIER_AUDIT_PATH)
print("Missing-feature report:", MISSING_FEATURE_PATH)

print(
    "\nImportant: price created the target labels "
    "but will be excluded from model inputs."
)

Step 2 completed successfully.
Complete clean data: /content/drive/MyDrive/Amazon-Marketplace-Product-Intelligence-Platform-BJIT/data/processed/products_price_tier_clean.csv
Labelled training data: /content/drive/MyDrive/Amazon-Marketplace-Product-Intelligence-Platform-BJIT/data/processed/products_price_tier_labeled.csv
Threshold report: /content/drive/MyDrive/Amazon-Marketplace-Product-Intelligence-Platform-BJIT/reports/price_tier/price_tier_thresholds.csv
Tier distribution: /content/drive/MyDrive/Amazon-Marketplace-Product-Intelligence-Platform-BJIT/reports/price_tier/price_tier_distribution.csv
Group label audit: /content/drive/MyDrive/Amazon-Marketplace-Product-Intelligence-Platform-BJIT/reports/price_tier/price_tier_group_label_audit.csv
Missing-feature report: /content/drive/MyDrive/Amazon-Marketplace-Product-Intelligence-Platform-BJIT/reports/price_tier/price_tier_missing_features.csv

Important: price created the target labels but will be excluded from model inputs.
